# Step 5: Multi-App Tasks with Window Focus and OCR Fallback

**Topic:** Switching focus between two applications and reading text via OCR when no
accessible text layer exists, then acting with the Step 4 structured action loop

**Resources used:**
- PyGetWindow docs (pygetwindow.readthedocs.io)
- pytesseract docs (github.com/madmaze/pytesseract)
- Step 1 (`01-pyautogui-basics`), Step 2 (`02-screenshot-agent-loop`), and Step 4
  (`04-structured-actions`) from this repo, reused directly rather than rewritten

**Summary:** Adds window focus control and OCR text extraction, then wires both into
the Step 4 structured agent loop to move data from one app to another, ending in a
text-file-to-spreadsheet mini-project.


## Setup

Requires `GROQ_API_KEY` in a `.env` file, same as Steps 2 and 4. Also requires Tesseract
OCR installed separately from pytesseract itself, pytesseract is just a wrapper around
the `tesseract` binary, installing the pip package alone does not install the OCR engine.

In [1]:
!pip install pygetwindow pytesseract pillow pyautogui mss groq python-dotenv pydantic


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Tesseract itself (the OCR engine, not the Python wrapper) has to be installed
# separately, e.g. from https://github.com/UB-Mannheim/tesseract/wiki on Windows

import os
import io
import time
import json
import base64
from typing import Optional, Literal

import pygetwindow as gw
import pytesseract
from PIL import Image

import pyautogui
import mss
import mss.tools
from dotenv import load_dotenv
from groq import Groq, APIStatusError
from pydantic import BaseModel, Field, ValidationError

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])
VISION_MODEL = "qwen/qwen3.8-27b"

# Windows does not put Tesseract on PATH automatically, pytesseract has no way to find
# the binary without this explicit path, adjust to wherever you actually installed it
pytesseract.pytesseract.tesseract_cmd = r"D:\Tesseract\tesseract.exe"

## Section 1: Window focus handling with pygetwindow

`getAllTitles()` includes many empty-string entries for background/system windows that
can't be usefully targeted, those get filtered out. `activate()` alone fails on a
minimized window on Windows, it has to be restored first.

In [3]:
def list_open_windows():
    return [title for title in gw.getAllTitles() if title.strip()]


def get_window_by_title(partial_title):
    # substring match, returns a list since more than one window can match the fragment
    matches = gw.getWindowsWithTitle(partial_title)
    if not matches:
        raise ValueError(f"No window found matching '{partial_title}'")
    return matches[0]


def activate_window(window):
    if window.isMinimized:
        window.restore()  # activate() on a minimized window silently fails on Windows
    window.activate()
    time.sleep(0.3)  # give the OS a moment to actually raise it before the next call


def get_window_bbox(window):
    # absolute screen coordinates, used later to crop screenshots to just this window
    return (window.left, window.top, window.left + window.width, window.top + window.height)


def set_window_position(window, x, y):
    window.moveTo(x, y)


Test this section on its own, lists whatever windows are open right now.

In [4]:
for title in list_open_windows():
    print(title)


● multi_app_tasks.ipynb - 05-multi-app-tasks - Visual Studio Code
(3) Naach Meri Rani: Guru Randhawa Feat. Nora Fatehi | Tanishk Bagchi | Nikhita Gandhi | Bhushan Kumar - YouTube - Personal - Microsoft​ Edge
Home · UB-Mannheim/tesseract Wiki - Google Chrome
PyAutoGUI basics script for computer-use agent - Claude - Google Chrome
New Volume (D:) - File Explorer
Windows Input Experience
Program Manager


## Section 2: OCR fallback with pytesseract

`image_to_string` gives plain text with no position information, useful when you just
need the content. `image_to_data` gives per-word bounding boxes, this is what lets OCR
output feed back into click coordinates the same way Step 3's grounding benchmark scored
model predictions against bounding boxes, except here the "ground truth" comes from OCR
itself instead of a known widget geometry.

In [5]:
def capture_window_image(window):
    x1, y1, x2, y2 = get_window_bbox(window)
    with mss.MSS() as sct:
        region = {"left": x1, "top": y1, "width": x2 - x1, "height": y2 - y1}
        shot = sct.grab(region)
        png_bytes = mss.tools.to_png(shot.rgb, shot.size)
    return Image.open(io.BytesIO(png_bytes))


def extract_text(window):
    image = capture_window_image(window)
    return pytesseract.image_to_string(image)


def extract_text_with_boxes(window):
    image = capture_window_image(window)
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

    boxes = []
    for i, word in enumerate(data["text"]):
        if not word.strip():
            continue  # image_to_data returns many blank-text rows for layout regions, skip them
        x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
        boxes.append({
            "text": word,
            "confidence": data["conf"][i],
            # image_to_data coordinates are relative to the cropped window image,
            # convert back to absolute screen coordinates so PyAutoGUI can click them
            "center_x": window.left + x + w // 2,
            "center_y": window.top + y + h // 2,
        })
    return boxes


Test on whatever window title you give it, run Section 1's list first to see valid titles.

In [8]:
test_window = get_window_by_title("Notepad")  # replace with any open window's title
activate_window(test_window)

print("Plain text:")
print(extract_text(test_window))

print("\nWord-level boxes:")
for box in extract_text_with_boxes(test_window)[:10]:
    print(box)


Plain text:
Untitled

File Edit View

Ln1, Col1 0 characters Plain text Windows (CRLF)



Word-level boxes:
{'text': 'Untitled', 'confidence': 95, 'center_x': 279, 'center_y': 190}
{'text': 'File', 'confidence': 96, 'center_x': 198, 'center_y': 238}
{'text': 'Edit', 'confidence': 91, 'center_x': 274, 'center_y': 238}
{'text': 'View', 'confidence': 67, 'center_x': 356, 'center_y': 238}
{'text': 'Ln1,', 'confidence': 77, 'center_x': 212, 'center_y': 848}
{'text': 'Col1', 'confidence': 77, 'center_x': 253, 'center_y': 847}
{'text': '0', 'confidence': 86, 'center_x': 324, 'center_y': 847}
{'text': 'characters', 'confidence': 92, 'center_x': 375, 'center_y': 847}
{'text': 'Plain', 'confidence': 96, 'center_x': 651, 'center_y': 847}
{'text': 'text', 'confidence': 96, 'center_x': 690, 'center_y': 848}


## Section 3: Structured action loop, reused from Step 4

This is the same `AgentAction` schema, forced tool call, and validation/retry logic from
`04-structured-actions/structured_actions.ipynb`, copied here rather than imported since
each step folder in this repo is self-contained. It is what Step 4 built on top of
Step 2's original screenshot-ask-execute loop, so this is that same loop, two steps
removed, not a new one.

In [9]:
class AgentAction(BaseModel):
    action: Literal["click", "type", "scroll", "wait", "done"] = Field(
        description="The single next action to perform"
    )
    x: Optional[int] = Field(default=None, description="X coordinate for click actions")
    y: Optional[int] = Field(default=None, description="Y coordinate for click actions")
    text: Optional[str] = Field(default=None, description="Text to type for type actions")
    direction: Optional[Literal["up", "down"]] = Field(default=None, description="Scroll direction")
    amount: Optional[int] = Field(default=None, description="Scroll amount in clicks")
    reasoning: str = Field(description="Short reason for choosing this action")


ACTION_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "submit_action",
        "description": "Submit the single next action to perform on the screen",
        "parameters": AgentAction.model_json_schema(),
    },
}


def capture_screenshot_base64():
    with mss.MSS() as sct:
        monitor = sct.monitors[1]
        shot = sct.grab(monitor)
        png_bytes = mss.tools.to_png(shot.rgb, shot.size)
    return base64.b64encode(png_bytes).decode("utf-8")


def get_validated_action(base64_image, task_description, history=None, max_retries=3):
    history = history or []
    temperature = 0.2
    screen_width, screen_height = pyautogui.size()

    for attempt in range(1, max_retries + 1):
        prompt = (
            f"You control a computer by looking at screenshots. The screen is "
            f"{screen_width}x{screen_height} pixels, origin (0,0) is the top-left corner. "
            f"Task: {task_description}\nSteps taken so far: {history}\n"
            "Call the submit_action tool with the single next action to take."
        )
        try:
            response = client.chat.completions.create(
                model=VISION_MODEL,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}},
                    ],
                }],
                tools=[ACTION_TOOL_SCHEMA],
                tool_choice={"type": "function", "function": {"name": "submit_action"}},
                temperature=temperature,
                max_completion_tokens=512,
            )
        except APIStatusError:
            temperature = max(temperature - 0.1, 0.0)
            continue

        message = response.choices[0].message
        if not message.tool_calls:
            temperature = max(temperature - 0.1, 0.0)
            continue

        try:
            raw_args = json.loads(message.tool_calls[0].function.arguments)
            return AgentAction.model_validate(raw_args)
        except (json.JSONDecodeError, ValidationError):
            temperature = max(temperature - 0.1, 0.0)
            continue

    raise RuntimeError(f"Failed to get a valid structured action after {max_retries} attempts")


def execute_structured_action(action: AgentAction):
    screen_width, screen_height = pyautogui.size()

    if action.action == "done":
        print("Model signaled done:", action.reasoning)
        return True

    if action.action == "click":
        if action.x is None or action.y is None or not (0 <= action.x <= screen_width and 0 <= action.y <= screen_height):
            print("Skipping click, invalid coordinates:", action)
            return False
        pyautogui.moveTo(action.x, action.y, duration=0.3)
        pyautogui.click()

    elif action.action == "type":
        if action.text:
            pyautogui.write(action.text, interval=0.05)

    elif action.action == "scroll":
        clicks = action.amount or 1
        pyautogui.scroll(clicks if action.direction == "up" else -clicks)

    elif action.action == "wait":
        time.sleep(1)

    return False


def run_structured_agent_loop(task_description, max_steps=15, delay_between_steps=1.0):
    history = []
    for step in range(1, max_steps + 1):
        b64_image = capture_screenshot_base64()
        action = get_validated_action(b64_image, task_description, history)
        print(f"Step {step}: {action.action} -> {action.reasoning}")
        history.append(action.model_dump())

        if execute_structured_action(action):
            print(f"Loop finished after {step} step(s)")
            return history
        time.sleep(delay_between_steps)

    print(f"Hit max_steps ({max_steps}) without the model signaling done")
    return history


## Section 4: The full multi-app task loop

`read_data_from_source` prefers a direct file read when a real file path is available,
since that is strictly more reliable than OCR, OCR is the fallback for apps that don't
expose their content any other way, not the default path when something better exists.
`write_data_to_target` hands off to Step 4's agent loop rather than hardcoding a click
position, since the target app's exact layout (which cell, which field) is not known in
advance.

In [10]:
def read_data_from_source(window_title, source_file_path=None):
    window = get_window_by_title(window_title)
    activate_window(window)

    if source_file_path and os.path.exists(source_file_path):
        # direct read is available and more reliable than OCR, use it instead of
        # screenshotting a window whose content we can already read from disk
        with open(source_file_path, "r") as f:
            return f.read().strip()

    # no direct access to the content, fall back to OCR on the window itself
    return extract_text(window).strip()


def write_data_to_target(window_title, text_to_write, task_template, max_steps=10):
    window = get_window_by_title(window_title)
    activate_window(window)

    task_description = task_template.format(text=text_to_write)
    return run_structured_agent_loop(task_description, max_steps=max_steps)


def run_multi_app_task(source_title, target_title, target_task_template, source_file_path=None):
    data = read_data_from_source(source_title, source_file_path=source_file_path)
    print(f"Read from '{source_title}': {data!r}")

    history = write_data_to_target(target_title, data, target_task_template)
    return data, history


## Section 5: Mini project, text file to spreadsheet

Creates a real sample text file so this cell is runnable without you needing to prepare
one first. Notepad has no accessible text layer this notebook reads from directly (no
DOM, no API call made here to pull its content), the only way to see the sample value in
this notebook is by reading the file straight off disk since we wrote it ourselves.
`Section 2's OCR` is validated separately below on the same Notepad window, to prove the
fallback path actually works, not just to assert it exists.

In [11]:
sample_file_path = os.path.join(os.getcwd(), "sample_data.txt")
with open(sample_file_path, "w") as f:
    f.write("42891")

os.startfile(sample_file_path)  # opens in the default .txt handler, Notepad on most Windows setups
time.sleep(1.5)  # let Notepad actually launch and render before anything tries to find its window


Confirm OCR can independently read the same value straight from the Notepad window, as a real fallback test, not an assumption.

In [12]:
notepad_window = get_window_by_title("Notepad")
activate_window(notepad_window)
ocr_result = extract_text(notepad_window).strip()
print(f"OCR read: {ocr_result!r}")


OCR read: 'Untitled\n\nFile Edit View\n\nLn1, Col1 0 characters Plain text Windows (CRLF)'


Open the spreadsheet app the same way Step 2's calculator mini-project opened the calculator, via the OS search shortcut, no hardcoded icon coordinates.

In [20]:
import webbrowser

# this URL creates a brand new blank spreadsheet directly, skipping the extra
# clicks of going through sheets.google.com's file picker first
webbrowser.open("https://docs.google.com/spreadsheets/create", new=1)
time.sleep(4)  # first load is slower than a local app, give the browser and page time to render

Run the full task: read the sample value (direct file read, the reliable path) and have the agent loop enter it into the spreadsheet's first cell.

In [21]:
spreadsheet_task_template = (
    "This is a Google Sheets spreadsheet open in a browser. Click cell A1, "
    "type '{text}', then press enter. Respond with action done once the "
    "value is visible in the cell."
)

value_read, task_history = run_multi_app_task(
    source_title="Notepad",
    target_title="Google Sheets",
    target_task_template=spreadsheet_task_template,
    source_file_path=sample_file_path,
)

Read from 'Notepad': '42891'
Step 1: click -> Click cell A1 to select it before typing
Step 2: type -> Type the value 42891 into the selected cell A1 and press Enter to confirm
Step 3: click -> Click cell A1 to select it before typing
Step 4: type -> Type the value 42891 into the selected cell A1 and press Enter to confirm
Step 5: click -> Click cell A1 to select it before typing
Step 6: type -> Type the value 42891 into the selected cell A1 and press Enter to confirm
Step 7: type -> Type the value 42891 into the selected cell A1 and press Enter to confirm
Step 8: wait -> The previous type actions did not include the text parameter, so the value was never entered. I need to wait for the screen to settle before retrying with the correct text.
Step 9: click -> Click cell A1 to select it before typing
Step 10: type -> Type the value 42891 into the selected cell A1 and press Enter to confirm
Hit max_steps (10) without the model signaling done


In [19]:
print(list_open_windows())

['● multi_app_tasks.ipynb - 05-multi-app-tasks - Visual Studio Code', 'PyAutoGUI basics script for computer-use agent - Claude - Google Chrome', 'Untitled - Notepad', '(3) Wada Raha Pyar Se Pyar Ka Lyrical Video | Khakee |Shreya Ghoshal|Akshay Kumar,Aishwarya Rai Bachchan - YouTube and 2 more pages - Personal - Microsoft\u200b Edge', 'Home · UB-Mannheim/tesseract Wiki - Google Chrome', 'New Volume (D:) - File Explorer', 'Windows Input Experience', 'Program Manager']
